Baseline Inference

In this notebook, I will run the base instruction-tuned model on the test set before fine-tuning.

The goal is to save the model's original outputs so I can later compare:

- expected output
- base model output
- fine-tuned model output

This helps us understand whether fine-tuning actually improves the rewriting task.

# Importing Libraries

In [ ]:
import json
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

# Configuration setup

## Set Path

In [2]:
test_path = Path("../data/processed/test.jsonl")

outputs_dir = Path("../outputs")

outputs_dir.mkdir(parents=True, exist_ok=True)

baseline_outputs_path  = outputs_dir / "baseline_outputs.jsonl"
baseline_preview_path = outputs_dir / "baseline_preview.csv"

print(f"Test path: {test_path} | test_exists: {test_path.exists()}")
print(f"Outputs directory: {outputs_dir} | outputs_dir_exists: {outputs_dir.exists()}")
print(f"Baseline outputs path: {baseline_outputs_path} | baseline_outputs_exists: {baseline_outputs_path.exists()}")
print(f"Baseline preview path: {baseline_preview_path} | baseline_preview_exists: {baseline_preview_path.exists()}")


Test path: ..\data\processed\test.jsonl | test_exists: True
Outputs directory: ..\outputs | outputs_dir_exists: True
Baseline outputs path: ..\outputs\baseline_outputs.jsonl | baseline_outputs_exists: False
Baseline preview path: ..\outputs\baseline_preview.csv | baseline_preview_exists: False


## load the dataset

In [3]:
test_rows = []

with test_path.open("r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line: test_rows.append(json.loads(line))

len(test_rows)


30

In [4]:
test_rows[0]

{'id': 'ex_0017',
 'category': 'workplace',
 'instruction': 'Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.',
 'input': 'I would appreciate it if you could review this section before the end of the day.',
 'output': 'Could you review this section by the end of the day?',
 'source': 'synthetic_curated'}

In [5]:
example = test_rows[0]

print("ID:", example["id"])
print("Category:", example["category"])

print("\nInput:")
print(example["input"])

print("\nExpected output:")
print(example["output"])

ID: ex_0017
Category: workplace

Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?


In [6]:
test_df = pd.DataFrame(test_rows)
test_df.head()

,id,category,instruction,input,output,source
0,ex_0017,workplace,"Rewrite this message so it sounds natural, cle...",I would appreciate it if you could review this...,Could you review this section by the end of th...,synthetic_curated
1,ex_0040,student_message,"Rewrite this message so it sounds natural, cle...",I wanted to ask whether there will be any offi...,Will you be holding office hours this week?,synthetic_curated
2,ex_0247,awkward_casual,"Rewrite this message so it sounds natural, cle...",I think we need to talk regarding the presenta...,I think we need to talk about the presentation...,synthetic_curated
3,ex_0005,workplace,"Rewrite this message so it sounds natural, cle...",Kindly note that I have updated the spreadshee...,I updated the spreadsheet with the latest info...,synthetic_curated
4,ex_0282,clearer_text,"Rewrite this message so it sounds natural, cle...",I am trying to say that the deadline is diffic...,I’m saying the deadline is difficult because I...,synthetic_curated


## Category distribution in test set

In [7]:
test_df["category"].value_counts()

category
workplace           6
student_message     4
wordy_to_concise    4
clearer_text        3
awkward_casual      3
follow_up           3
formal_email        3
polite_request      2
reminder            2
Name: count, dtype: int64

## select a tiny subset

In [8]:
preview_rows = test_rows[:5]

for row in preview_rows:
    print("=" * 80)
    print("ID:", row["id"])
    print("Category:", row["category"])
    print("Input:", row["input"])
    print("Expected:", row["output"])


ID: ex_0017
Category: workplace
Input: I would appreciate it if you could review this section before the end of the day.
Expected: Could you review this section by the end of the day?
ID: ex_0040
Category: student_message
Input: I wanted to ask whether there will be any office hours available this week.
Expected: Will you be holding office hours this week?
ID: ex_0247
Category: awkward_casual
Input: I think we need to talk regarding the presentation slides.
Expected: I think we need to talk about the presentation slides.
ID: ex_0005
Category: workplace
Input: Kindly note that I have updated the spreadsheet with the latest information.
Expected: I updated the spreadsheet with the latest information.
ID: ex_0282
Category: clearer_text
Input: I am trying to say that the deadline is difficult because of other submissions.
Expected: I’m saying the deadline is difficult because I have other submissions.


Why only 5 first?
- Before we run an LLM on all 30 test examples, we’ll test the generation code on 5 examples.
- Tiny batches first. Less chaos. Fewer “why is my laptop screaming” moments.

Test Set Loaded

We loaded the held-out test set and inspected a small preview batch.

The test set will be used for baseline inference before fine-tuning. Later, we will run the fine-tuned model on the same examples and compare the outputs side by side.

In [9]:
device = "cpu"
device


'cpu'

# Run the model

## Set the model name

In [10]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct" #name of a pre-trained language model
model_name

'Qwen/Qwen2.5-0.5B-Instruct'

## Load the tokenizer

In [11]:
#The tokenizer converts text into token IDs that the model understands.
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer

Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-0.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=Fa

## load the base model

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype= torch.float32,
    # device_map="auto" if device == "cuda" else None
)

# if device == "cpu":
model = model.to(device)

model.eval()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

## Build the prompt

In [13]:
def build_prompt(message):
    return f"""Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.
Message: {message}
Rewritten message:
            """

## test on one example

In [14]:
# example = test_rows[0]
# prompt = build_prompt(example["input"])
# print(prompt)

Rewrite this message so it sounds natural, clear, and human while keeping the same meaning.
Message: I would appreciate it if you could review this section before the end of the day.
Rewritten message:
            


## Generate one baseline output

In [16]:
# inputs = tokenizer(prompt, return_tensors="pt").to(device)

# with torch.no_grad():
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=80,
#         do_sample=False,
#         pad_token_id=tokenizer.eos_token_id
#     )

# generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
# generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)

# print("Input:")
# print(example["input"])

# print("\nExpected output:")
# print(example["output"])

# print("\nCategory:")
# print(example["category"])

# print("\nBase model output:")
# print(generated_text.strip())

Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?

Category:
workplace

Base model output:
I hope to have a chance to review this section by the time we conclude our meeting at 4 PM. 

This version maintains the original meaning but uses more conversational language and is easier to understand for a general audience. It also includes an option to include the date in the review, which can be helpful for those who are planning their schedule around the meeting. The revised message is clearer and more


First Baseline Generation

We loaded the base instruction-tuned model and generated one rewrite before fine-tuning.

This output is important because it gives us the model's original behavior on the task. Later, we will compare this with the fine-tuned model's output on the same test examples.

## Adding Cleaning Function

In [50]:
def clean_generated_output(text):

    # Keep only the first generated rewrite line
    text = str(text).strip()

    if not text:
        return ""
    
    lines = [line.strip() for line in text.split("\n") if line.strip()]

    if not lines:
        return ""
    
    return lines[0]

    

## Generation Function

In [51]:
def generate_rewrite(message, max_new_tokens=80):
    prompt = build_prompt(message)
    
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    raw_generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    cleaned_generated_text = clean_generated_output(raw_generated_text)

    return raw_generated_text, cleaned_generated_text

## Test the model

In [61]:
# test the function on a single example
example = test_rows[0]

rewritten_message, cleaned_generated_text = generate_rewrite(example["input"])

print("\nCategory:")
print(example["category"])

print("Input:")
print(example["input"])

print("\nExpected output:")
print(example["output"])

print("\nRewritten message:")
print(cleaned_generated_text)




Category:
workplace
Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?

Rewritten message:
I hope to have a chance to review this section by the time we conclude our meeting at 4 PM.


In [21]:
# preview_results= []

# for row in tqdm(preview_rows):
#     baseoutput = generate_rewrite(row["input"])

#     preview_results.append(
#         {
#             "id":row["id"],
#             "category": row["category"],
#             "input": row["input"],
#             "expected": row["output"],
#             "baseline_output": baseoutput
#         }
#     )

100%|██████████| 5/5 [00:46<00:00,  9.39s/it]


In [22]:
# len(preview_results)

5

In [23]:
# preview_df = pd.DataFrame(preview_results)

# preview_df

,id,category,input,expected,baseline_output
0,ex_0017,workplace,I would appreciate it if you could review this...,Could you review this section by the end of th...,I hope to have a chance to review this section...
1,ex_0040,student_message,I wanted to ask whether there will be any offi...,Will you be holding office hours this week?,I would like to inquire about the availability...
2,ex_0247,awkward_casual,I think we need to talk regarding the presenta...,I think we need to talk about the presentation...,We should discuss the presentation slides. \n\...
3,ex_0005,workplace,Kindly note that I have updated the spreadshee...,I updated the spreadsheet with the latest info...,Please review the most recent updates to the s...
4,ex_0282,clearer_text,I am trying to say that the deadline is diffic...,I’m saying the deadline is difficult because I...,I'm trying to convey that the deadline is chal...


In [26]:
# for result in preview_results:
#     print("=" * 100)
#     print("ID:", result["id"])
#     print("Category:", result["category"])
    
#     print("\nInput:")
#     print(result["input"])
    
#     print("\nExpected output:")
#     print(result["expected"])
    
#     print("\nBase model output:")
#     print(result["baseline_output"])
#     print()

ID: ex_0017
Category: workplace

Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?

Base model output:
I hope to have a chance to review this section by the time we conclude our meeting at 4 PM. 

This version maintains the original meaning but uses more conversational language and is easier to understand for a general audience. It also includes an option to include the date in the review, which can be helpful for those who are planning their schedule around the meeting. The revised message is clearer and more

ID: ex_0040
Category: student_message

Input:
I wanted to ask whether there will be any office hours available this week.

Expected output:
Will you be holding office hours this week?

Base model output:
I would like to inquire about the availability of office hours for this week. 

This version maintains the original meaning but uses more conversational language and avo

Baseline Preview Results

I generated baseline rewrites for 5 test examples using the base model.
These results help us inspect the model's behavior before fine-tuning.
At this stage, we are looking for patterns such as:

- whether the base model keeps the meaning - Almost
- whether it makes the text more natural - Not much
- whether it becomes too formal - Yes
- whether it adds extra information - Yes
- whether it copies the input too closely - Yes

## Run the baseline inference on full test set

In [62]:
baseline_results = []

for row in tqdm(test_rows):
    raw_output, cleaned_output = generate_rewrite(row["input"])

    baseline_results.append(
            {
                "id":row["id"],
                "category": row["category"],
                "instruction":row["instruction"],
                "input": row["input"],
                "expected": row["output"],
                "baseline_output": cleaned_output,
                "raw_baseline_output": raw_output
            }
        )

len(baseline_results)

100%|██████████| 30/30 [04:08<00:00,  8.27s/it]


30

In [71]:
baseline_df = pd.DataFrame(baseline_results)
baseline_df.head()

,id,category,instruction,input,expected,baseline_output,raw_baseline_output
0,ex_0017,workplace,"Rewrite this message so it sounds natural, cle...",I would appreciate it if you could review this...,Could you review this section by the end of th...,I hope to have a chance to review this section...,I hope to have a chance to review this section...
1,ex_0040,student_message,"Rewrite this message so it sounds natural, cle...",I wanted to ask whether there will be any offi...,Will you be holding office hours this week?,I would like to inquire about the availability...,I would like to inquire about the availability...
2,ex_0247,awkward_casual,"Rewrite this message so it sounds natural, cle...",I think we need to talk regarding the presenta...,I think we need to talk about the presentation...,We should discuss the presentation slides.,We should discuss the presentation slides. \n\...
3,ex_0005,workplace,"Rewrite this message so it sounds natural, cle...",Kindly note that I have updated the spreadshee...,I updated the spreadsheet with the latest info...,Please review the most recent updates to the s...,Please review the most recent updates to the s...
4,ex_0282,clearer_text,"Rewrite this message so it sounds natural, cle...",I am trying to say that the deadline is diffic...,I’m saying the deadline is difficult because I...,I'm trying to convey that the deadline is chal...,I'm trying to convey that the deadline is chal...


In [72]:
with baseline_outputs_path.open("w",encoding="utf-8") as file:
    for row in baseline_results:
        file.write(json.dumps(row,ensure_ascii=False) + "\n")
baseline_outputs_path

WindowsPath('../outputs/baseline_outputs.jsonl')

In [73]:
baseline_df.to_csv(baseline_preview_path, index=False, encoding="utf-8")
baseline_preview_path

WindowsPath('../outputs/baseline_preview.csv')

In [74]:
print("JSONL saved:", baseline_outputs_path.exists())
print("CSV saved:", baseline_preview_path.exists())

JSONL saved: True
CSV saved: True


In [75]:
for result in baseline_results[:3]:
    print("=" * 100)
    print("ID:", result["id"])
    print("Category:", result["category"])

    print("\nInput:")
    print(result["input"])

    print("\nExpected output:")
    print(result["expected"])

    print("\nBase model output:")
    print(result["baseline_output"])
    print()

ID: ex_0017
Category: workplace

Input:
I would appreciate it if you could review this section before the end of the day.

Expected output:
Could you review this section by the end of the day?

Base model output:
I hope to have a chance to review this section by the time we conclude our meeting at 4 PM.

ID: ex_0040
Category: student_message

Input:
I wanted to ask whether there will be any office hours available this week.

Expected output:
Will you be holding office hours this week?

Base model output:
I would like to inquire about the availability of office hours for this week.

ID: ex_0247
Category: awkward_casual

Input:
I think we need to talk regarding the presentation slides.

Expected output:
I think we need to talk about the presentation slides.

Base model output:
We should discuss the presentation slides.



Full Baseline Inference Complete

I generated baseline outputs for all test examples using the base model before fine-tuning.

The results were saved as:

- `outputs/baseline_outputs.jsonl`
- `outputs/baseline_preview.csv`

These files will be used later to compare the base model against the fine-tuned model on the same held-out test examples.

# Evaluate

## Baseline output length

In [76]:
def count_words(text):
    return len(str(text).split())

In [ ]:
# baseline_df["input_word_count"] = baseline_df["input"].apply(count_words)
# baseline_df["expected_word_count"] = baseline_df["expected"].apply(count_words)
# baseline_df["base_output_word_count"] = baseline_df["baseline_output"].apply(count_words)

In [ ]:
# baseline_df["baseline_output"][0]

'I hope to have a chance to review this section by the time we conclude our meeting at 4 PM. \n\nThis version maintains the original meaning but uses more conversational language and is easier to understand for a general audience. It also includes an option to include the date in the review, which can be helpful for those who are planning their schedule around the meeting. The revised message is clearer and more'

We can see that the output has lot of unwanted sentences after the rewritten message which is counting towards the length so we need to re work on the generation function

In [78]:
baseline_df["input_word_count"] = baseline_df["input"].apply(count_words)
baseline_df["expected_word_count"] = baseline_df["expected"].apply(count_words)
baseline_df["baseline_output_word_count"] = baseline_df["baseline_output"].apply(count_words)
baseline_df["raw_baseline_output_word_count"] = baseline_df["raw_baseline_output"].apply(count_words)



In [77]:
baseline_df.columns

Index(['id', 'category', 'instruction', 'input', 'expected', 'baseline_output',
       'raw_baseline_output'],
      dtype='object')

In [79]:
baseline_df.head()

,id,category,instruction,input,expected,baseline_output,raw_baseline_output,input_word_count,expected_word_count,baseline_output_word_count,raw_baseline_output_word_count
0,ex_0017,workplace,"Rewrite this message so it sounds natural, cle...",I would appreciate it if you could review this...,Could you review this section by the end of th...,I hope to have a chance to review this section...,I hope to have a chance to review this section...,16,11,20,73
1,ex_0040,student_message,"Rewrite this message so it sounds natural, cle...",I wanted to ask whether there will be any offi...,Will you be holding office hours this week?,I would like to inquire about the availability...,I would like to inquire about the availability...,14,8,14,67
2,ex_0247,awkward_casual,"Rewrite this message so it sounds natural, cle...",I think we need to talk regarding the presenta...,I think we need to talk about the presentation...,We should discuss the presentation slides.,We should discuss the presentation slides. \n\...,10,10,6,66
3,ex_0005,workplace,"Rewrite this message so it sounds natural, cle...",Kindly note that I have updated the spreadshee...,I updated the spreadsheet with the latest info...,Please review the most recent updates to the s...,Please review the most recent updates to the s...,12,8,9,65
4,ex_0282,clearer_text,"Rewrite this message so it sounds natural, cle...",I am trying to say that the deadline is diffic...,I’m saying the deadline is difficult because I...,I'm trying to convey that the deadline is chal...,I'm trying to convey that the deadline is chal...,14,11,14,62


## Extra generated text

In [80]:
baseline_df["extra_generated_words"] = baseline_df["raw_baseline_output_word_count"] - baseline_df["baseline_output_word_count"]

## Summary Statistics for word counts

In [84]:
baseline_df[[
    "input_word_count",
    "expected_word_count",
    "baseline_output_word_count",
    "raw_baseline_output_word_count",
    "extra_generated_words"
]].describe()

,input_word_count,expected_word_count,baseline_output_word_count,raw_baseline_output_word_count,extra_generated_words
count,30.000000,30.000000,30.000000,30.000000,30.000000
mean,12.833333,9.600000,13.600000,67.166667,53.566667
std,1.782740,2.127002,4.148951,2.640315,4.264394
min,10.000000,5.000000,6.000000,62.000000,44.000000
25%,12.000000,8.000000,10.250000,65.250000,52.000000
50%,13.000000,10.000000,13.000000,66.000000,53.500000
75%,14.000000,11.000000,15.750000,69.000000,56.000000
max,16.000000,13.000000,22.000000,73.000000,61.000000


In [87]:
print("Average Word Counts")
print("-" * 30)
print("Input:", round(baseline_df["input_word_count"].mean(), 2))
print("Expected output:", round(baseline_df["expected_word_count"].mean(), 2))
print("Base model output:", round(baseline_df["baseline_output_word_count"].mean(), 2))
print("Raw base output:", round(baseline_df["raw_baseline_output_word_count"].mean(), 2))
print("Extra generated words:", round(baseline_df["extra_generated_words"].mean(), 2))

Average Word Counts
------------------------------
Input: 12.83
Expected output: 9.6
Base model output: 13.6
Raw base output: 67.17
Extra generated words: 53.57


## baseline warning flags

In [89]:
baseline_df["empty_base_output"] = baseline_df["baseline_output"].str.strip() == ""

baseline_df["base_same_as_input"] = (
    baseline_df["baseline_output"].str.strip().str.lower()
    == baseline_df["input"].str.strip().str.lower()
)

baseline_df["base_longer_than_input"] = (
    baseline_df["baseline_output_word_count"] > baseline_df["input_word_count"] + 10
)

baseline_df["base_very_short"] = (
    baseline_df["baseline_output_word_count"] < 3
)

baseline_df["raw_had_extra_text"] = (
    baseline_df["extra_generated_words"] > 0
)

baseline_df[[
    "id",
    "category",
    "empty_base_output",
    "base_same_as_input",
    "base_longer_than_input",
    "base_very_short",
    "raw_had_extra_text"
]].head()

,id,category,empty_base_output,base_same_as_input,base_longer_than_input,base_very_short,raw_had_extra_text
0,ex_0017,workplace,False,False,False,False,True
1,ex_0040,student_message,False,False,False,False,True
2,ex_0247,awkward_casual,False,False,False,False,True
3,ex_0005,workplace,False,False,False,False,True
4,ex_0282,clearer_text,False,False,False,False,True


In [90]:
print("Baseline Quality Warning Counts")
print("-" * 40)

print("Empty outputs:", baseline_df["empty_base_output"].sum())
print("Same as input:", baseline_df["base_same_as_input"].sum())
print("Longer than input by 10+ words:", baseline_df["base_longer_than_input"].sum())
print("Very short outputs:", baseline_df["base_very_short"].sum())
print("Raw outputs with extra text:", baseline_df["raw_had_extra_text"].sum())

Baseline Quality Warning Counts
----------------------------------------
Empty outputs: 0
Same as input: 0
Longer than input by 10+ words: 0
Very short outputs: 0
Raw outputs with extra text: 30


## Suspicious cleaned outputs

In [91]:
suspicious_baseline_df = baseline_df[
    baseline_df["empty_base_output"]
    | baseline_df["base_same_as_input"]
    | baseline_df["base_longer_than_input"]
    | baseline_df["base_very_short"]
]



In [94]:
suspicious_baseline_df[[
    "id",
    "category",
    "input",
    "expected",
    "raw_baseline_output",
    "baseline_output"
]]

,id,category,input,expected,raw_baseline_output,baseline_output


## Save updated quality review CSV

In [95]:
baseline_quality_review_path = outputs_dir / "baseline_quality_review.csv"

baseline_df.to_csv(baseline_quality_review_path, index=False)

baseline_quality_review_path

WindowsPath('../outputs/baseline_quality_review.csv')

Baseline Output Length and Quality Inspection

I inspected the base model outputs using both raw and cleaned generations.

The raw output preserves exactly what the model generated.
The cleaned output keeps only the rewritten message and removes extra generated text after the first rewrite line.

For analysis, I use the cleaned `baseline_output` field because it better represents the actual rewrite. I keep `raw_baseline_output` for debugging and traceability.

I checked:

- input length
- expected output length
- cleaned base model output length
- raw base model output length
- extra generated words
- empty outputs
- copied outputs
- overly long outputs
- very short outputs